# Task 3 Optuna screening
Attach extracted task2-delivery bundle. Each trial starts from the selected Task 2 weights. Eight trials, five total screening epochs (two warm-up + three joint). Baseline results stay separate. Run setup first.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# For a NEW search, ensure these new files were pushed. For resume, keep the same revision.
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)


In [ ]:
# Locate the complete selected Task 2 bundle (four named model folders).
candidates = [p.parent for p in Path('/kaggle/input').rglob('manifest.json')
              if all((p.parent / name / 'best.pt').is_file() for name in ['classifier','salt','blur','occlusion'])]
print(candidates)
assert len(candidates) == 1, 'Attach one extracted task2-delivery bundle, or set bundle to its exact folder.'
bundle = candidates[0]


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], cwd=repo, check=True)


In [ ]:
import shutil
search = Path('/kaggle/working/task3-optuna')
restore_folder = None # after reset: exact attached COMPLETE search folder containing study.json
if restore_folder is not None and not search.exists():
    source = Path(restore_folder)
    assert (source / 'study.json').is_file()
    shutil.copytree(source, search)
subprocess.run([sys.executable, '-u', '-m', 'optimization.optimize_task3',
    '--bundle', str(bundle), '--output-dir', str(search), '--device', 'cuda',
    '--trials', '8', '--screen-epochs', '5', '--max-hours', '2'], cwd=repo, check=True)


In [ ]:
import json
from IPython.display import display, FileLink
if (search / 'leaderboard.json').is_file():
    for trial in json.loads((search / 'leaderboard.json').read_text())['ranked_trials']:
        print(trial['number'], trial['score'], trial['params'])
backup = shutil.make_archive('/kaggle/working/task3-optuna_backup', 'zip', search.parent, search.name)
display(FileLink(backup))


The trial count is a TOTAL target. Rerun to continue an interrupted search; download the entire backup even if fewer than eight trials completed. Keep code, config, screen budget and single visible GPU unchanged. Winners remain provisional until continued to 10 total epochs with their original trial YAML and last.pt. Do not rerun the eight-trial search to continue a selected winner.
